# 8.3 Pruning and Sparsity — Apply

## Objective

Explore weight pruning and sparsity in ONNX models. You will:

1. Build dense models and create sparse versions by zeroing small weights
2. Measure sparsity ratios and their effect on model accuracy
3. Compare structured vs unstructured pruning
4. Benchmark dense vs sparse inference
5. Build an iterative magnitude pruning pipeline

**Key concept — magnitude pruning criterion:**

$$\text{mask}_{ij} = \begin{cases} 1 & \text{if } |W_{ij}| > \tau \\ 0 & \text{otherwise} \end{cases}$$

where $\tau$ is the pruning threshold, typically the $p$-th percentile of $|W|$.

**Sparsity ratio:**

$$\text{sparsity}(W) = \frac{\#\{W_{ij} = 0\}}{\#\{W_{ij}\}} = 1 - \frac{\text{nnz}(W)}{|W|}$$

In [ ]:
# Setup
import numpy as np
import onnx
from onnx import helper, TensorProto, numpy_helper
from onnx.checker import check_model
import onnxruntime as ort
import time
import os
import tempfile
import copy

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

TMPDIR = tempfile.mkdtemp(prefix='onnx_prune_')
print(f"onnx {onnx.__version__}, onnxruntime {ort.__version__}")
print(f"Temp dir: {TMPDIR}")

---
## Exercise 1: Build a Dense Model

We build a multi-layer perceptron with dense (non-sparse) weights.
This serves as our baseline for pruning experiments.

In [ ]:
np.random.seed(42)

def build_mlp(dims, batch=16, weight_arrays=None):
    """Build an MLP with given layer dimensions.
    If weight_arrays is provided, use those weights instead of random."""
    inits, nodes = [], []
    prev_name = 'X'

    for i in range(len(dims) - 1):
        d_in, d_out = dims[i], dims[i+1]
        if weight_arrays and i < len(weight_arrays):
            W = weight_arrays[i]
        else:
            W = np.random.randn(d_in, d_out).astype(np.float32) * np.sqrt(2.0/d_in)
        b = np.zeros(d_out, dtype=np.float32)
        inits.append(numpy_helper.from_array(W, f'W{i}'))
        inits.append(numpy_helper.from_array(b, f'b{i}'))
        nodes.append(helper.make_node('MatMul', [prev_name, f'W{i}'], [f'mm{i}']))
        nodes.append(helper.make_node('Add', [f'mm{i}', f'b{i}'], [f'add{i}']))
        if i < len(dims) - 2:
            nodes.append(helper.make_node('Relu', [f'add{i}'], [f'r{i}']))
            prev_name = f'r{i}'
        else:
            prev_name = f'add{i}'

    X_i = helper.make_tensor_value_info('X', TensorProto.FLOAT, [batch, dims[0]])
    Y_i = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [batch, dims[-1]])
    g = helper.make_graph(nodes, 'mlp', [X_i], [Y_i], initializer=inits)
    m = helper.make_model(g, opset_imports=[helper.make_opsetid('', 17)])
    check_model(m)
    return m

dims = [128, 256, 256, 64]
dense_model = build_mlp(dims)

# Analyze weight statistics
total_params = 0
for init in dense_model.graph.initializer:
    arr = numpy_helper.to_array(init)
    if arr.ndim == 2:  # weight matrices only
        total_params += arr.size
        print(f"  {init.name}: shape={arr.shape}, "
              f"mean={arr.mean():.4f}, std={arr.std():.4f}, "
              f"min={arr.min():.4f}, max={arr.max():.4f}")

print(f"\nTotal weight parameters: {total_params:,}")
print(f"Dense model nodes: {len(dense_model.graph.node)}")

---
## Exercise 2: Create a Sparse Version (Magnitude Pruning)

Magnitude pruning zeros out weights below a threshold. For target sparsity $p$:

$$\tau = \text{percentile}(|W|, 100p)$$

$$W_{\text{pruned}} = W \odot \text{mask}, \quad \text{mask}_{ij} = \mathbb{1}[|W_{ij}| > \tau]$$

The key insight: neural network weights follow approximately Gaussian distributions,
so many weights are near zero and contribute minimally to the output.

In [ ]:
def prune_model(model, target_sparsity=0.5):
    """Apply magnitude-based unstructured pruning to all weight matrices."""
    pruned = copy.deepcopy(model)
    stats = []

    for init in pruned.graph.initializer:
        arr = numpy_helper.to_array(init)
        if arr.ndim < 2:
            continue

        threshold = np.percentile(np.abs(arr), target_sparsity * 100)
        mask = (np.abs(arr) > threshold).astype(np.float32)
        pruned_arr = arr * mask

        actual_sparsity = 1.0 - np.count_nonzero(pruned_arr) / pruned_arr.size
        stats.append({
            'name': init.name,
            'shape': arr.shape,
            'threshold': threshold,
            'sparsity': actual_sparsity,
            'nnz': np.count_nonzero(pruned_arr),
            'total': pruned_arr.size,
        })

        new_init = numpy_helper.from_array(pruned_arr, init.name)
        init.CopyFrom(new_init)

    return pruned, stats

# Prune at 50% sparsity
sparse_model_50, stats_50 = prune_model(dense_model, target_sparsity=0.5)
check_model(sparse_model_50)

print(f"Magnitude Pruning at 50% target sparsity:")
print(f"{'Layer':<10} {'Shape':>12} {'Threshold':>10} {'Sparsity':>10} {'NNZ':>8} {'Total':>8}")
print('-' * 62)
for s in stats_50:
    shape_str = f"{s['shape'][0]}x{s['shape'][1]}"
    print(f"{s['name']:<10} {shape_str:>12} {s['threshold']:>10.4f} {s['sparsity']:>9.1%} {s['nnz']:>8} {s['total']:>8}")

---
## Exercise 3: Compare Dense vs Sparse Inference Outputs

Pruning introduces error proportional to the pruned weight magnitudes.
The output perturbation from pruning weight matrix $W$ by $\Delta W$ is:

$$\|\Delta y\| \leq \|\Delta W\| \cdot \|x\|$$

We measure this empirically across multiple inputs.

In [ ]:
sess_dense  = ort.InferenceSession(dense_model.SerializeToString(),
                                   providers=['CPUExecutionProvider'])
sess_sparse = ort.InferenceSession(sparse_model_50.SerializeToString(),
                                   providers=['CPUExecutionProvider'])

n_samples = 200
abs_diffs = []
rel_diffs = []

for _ in range(n_samples):
    x = np.random.randn(16, 128).astype(np.float32)
    y_dense  = sess_dense.run(None, {'X': x})[0]
    y_sparse = sess_sparse.run(None, {'X': x})[0]
    abs_diffs.append(np.mean(np.abs(y_dense - y_sparse)))
    denom = np.abs(y_dense) + 1e-8
    rel_diffs.append(np.mean(np.abs(y_dense - y_sparse) / denom))

abs_diffs = np.array(abs_diffs)
rel_diffs = np.array(rel_diffs)

print(f"Dense vs Sparse (50% sparsity) — {n_samples} samples:")
print(f"  Mean abs error:  {abs_diffs.mean():.6f}")
print(f"  P95 abs error:   {np.percentile(abs_diffs, 95):.6f}")
print(f"  Mean rel error:  {rel_diffs.mean():.4%}")
print(f"  P95 rel error:   {np.percentile(rel_diffs, 95):.4%}")

# Compare at multiple sparsity levels
print(f"\n{'Sparsity':>10} {'Mean |err|':>12} {'Mean rel%':>10}")
print('-' * 35)
for sp in [0.1, 0.3, 0.5, 0.7, 0.9, 0.95]:
    sp_model, _ = prune_model(dense_model, target_sparsity=sp)
    sess_sp = ort.InferenceSession(sp_model.SerializeToString(),
                                   providers=['CPUExecutionProvider'])
    errs = []
    rels = []
    for _ in range(50):
        x = np.random.randn(16, 128).astype(np.float32)
        y_d = sess_dense.run(None, {'X': x})[0]
        y_s = sess_sp.run(None, {'X': x})[0]
        errs.append(np.mean(np.abs(y_d - y_s)))
        rels.append(np.mean(np.abs(y_d - y_s) / (np.abs(y_d) + 1e-8)))
    print(f"{sp:>9.0%} {np.mean(errs):>12.6f} {np.mean(rels):>9.2%}")

---
## Exercise 4: Benchmark Dense vs Sparse Performance

While ONNX Runtime doesn't natively exploit unstructured sparsity
(most benefit requires specialized sparse kernels), we can measure
any indirect effects from simpler weight patterns.

In [ ]:
def benchmark(model, input_shape, n_warmup=100, n_runs=1000):
    """Benchmark inference latency."""
    sess = ort.InferenceSession(model.SerializeToString(),
                                providers=['CPUExecutionProvider'])
    x = np.random.randn(*input_shape).astype(np.float32)
    name = sess.get_inputs()[0].name

    for _ in range(n_warmup):
        sess.run(None, {name: x})

    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        sess.run(None, {name: x})
        times.append((time.perf_counter() - t0) * 1000)
    return np.array(times)

t_dense = benchmark(dense_model, (16, 128))

sparsity_levels = [0.0, 0.3, 0.5, 0.7, 0.9, 0.95]
bench_results = [{'sparsity': 0.0, 'mean_ms': t_dense.mean(), 'p95_ms': np.percentile(t_dense, 95)}]

for sp in sparsity_levels[1:]:
    sp_model, _ = prune_model(dense_model, sp)
    t_sp = benchmark(sp_model, (16, 128))
    bench_results.append({
        'sparsity': sp,
        'mean_ms': t_sp.mean(),
        'p95_ms': np.percentile(t_sp, 95),
    })

print(f"{'Sparsity':>10} {'Mean (ms)':>10} {'P95 (ms)':>10} {'vs Dense':>10}")
print('-' * 42)
base = bench_results[0]['mean_ms']
for r in bench_results:
    ratio = base / r['mean_ms']
    print(f"{r['sparsity']:>9.0%} {r['mean_ms']:>10.4f} {r['p95_ms']:>10.4f} {ratio:>9.2f}x")

print(f"\nNote: Without sparse kernel support in the runtime, ")
print(f"unstructured sparsity may not improve latency significantly.")
print(f"The real benefit comes from reduced model size and memory bandwidth.")

---
## Exercise 5: Measure Sparsity Ratio and Weight Distribution

Visualize how weight magnitudes distribute and where the pruning threshold falls.
For Gaussian-initialized weights $W \sim \mathcal{N}(0, \sigma^2)$:

$$P(|W| \leq \tau) = \text{erf}\!\left(\frac{\tau}{\sigma\sqrt{2}}\right)$$

So pruning at 50% sparsity removes weights within ~0.67$\sigma$ of zero.

In [ ]:
def compute_sparsity_profile(model):
    """Compute per-layer and global sparsity statistics."""
    results = []
    all_weights = []

    for init in model.graph.initializer:
        arr = numpy_helper.to_array(init)
        if arr.ndim < 2:
            continue
        nnz = np.count_nonzero(arr)
        total = arr.size
        sparsity = 1.0 - nnz / total
        results.append({
            'name': init.name,
            'shape': arr.shape,
            'nnz': nnz,
            'total': total,
            'sparsity': sparsity,
            'mean_abs': np.mean(np.abs(arr)),
            'std': np.std(arr),
        })
        all_weights.append(arr.flatten())

    all_w = np.concatenate(all_weights)
    global_sparsity = 1.0 - np.count_nonzero(all_w) / all_w.size
    return results, all_w, global_sparsity

# Profile the 50% sparse model
profile, all_weights, global_sp = compute_sparsity_profile(sparse_model_50)

print(f"Sparsity Profile (50% pruned model):")
print(f"{'Layer':<8} {'Shape':>12} {'Sparsity':>10} {'NNZ':>8} {'MeanAbs':>8} {'Std':>8}")
print('-' * 58)
for p in profile:
    shape_str = f"{p['shape'][0]}x{p['shape'][1]}"
    print(f"{p['name']:<8} {shape_str:>12} {p['sparsity']:>9.1%} {p['nnz']:>8} {p['mean_abs']:>8.4f} {p['std']:>8.4f}")
print(f"\nGlobal sparsity: {global_sp:.1%}")

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # Weight magnitude histogram
    ax1.hist(np.abs(all_weights[all_weights != 0]), bins=50, color='steelblue',
             edgecolor='black', alpha=0.8, label='Non-zero weights')
    threshold_50 = np.percentile(np.abs(all_weights[all_weights != 0]), 0)  # all remaining
    ax1.set_xlabel('|Weight|')
    ax1.set_ylabel('Count')
    ax1.set_title('Weight Magnitude Distribution (after pruning)', fontweight='bold')
    ax1.legend()
    ax1.grid(alpha=0.3)

    # Sparsity per layer
    names = [p['name'] for p in profile]
    sparsities = [p['sparsity'] for p in profile]
    ax2.barh(names, sparsities, color='coral', edgecolor='black')
    ax2.set_xlabel('Sparsity Ratio')
    ax2.set_title('Per-Layer Sparsity', fontweight='bold')
    ax2.set_xlim(0, 1)
    for i, s in enumerate(sparsities):
        ax2.text(s + 0.01, i, f'{s:.1%}', va='center', fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.show()

---
## Exercise 6: Structured vs Unstructured Sparsity

**Unstructured:** individual weights zeroed (fine-grained, high accuracy, hard to accelerate)

**Structured:** entire rows/columns/filters zeroed (coarser, lower accuracy, easy to accelerate)

For structured pruning of rows in $W \in \mathbb{R}^{d_{\text{in}} \times d_{\text{out}}}$, we use the
$\ell_2$-norm criterion:

$$\text{importance}(\text{row}_j) = \|W_{j,:}\|_2 = \sqrt{\sum_k W_{jk}^2}$$

In [ ]:
def structured_prune(model, target_sparsity=0.5):
    """Apply structured pruning: zero out entire output columns by L2 norm."""
    pruned = copy.deepcopy(model)
    stats = []

    for init in pruned.graph.initializer:
        arr = numpy_helper.to_array(init)
        if arr.ndim != 2:
            continue

        # Compute L2 norm of each output column
        col_norms = np.linalg.norm(arr, axis=0)  # shape: (d_out,)
        n_prune = int(target_sparsity * len(col_norms))
        prune_indices = np.argsort(col_norms)[:n_prune]

        mask = np.ones_like(arr)
        mask[:, prune_indices] = 0
        pruned_arr = arr * mask

        actual_sp = 1.0 - np.count_nonzero(pruned_arr) / pruned_arr.size
        stats.append({
            'name': init.name,
            'cols_pruned': n_prune,
            'cols_total': arr.shape[1],
            'element_sparsity': actual_sp,
        })

        new_init = numpy_helper.from_array(pruned_arr, init.name)
        init.CopyFrom(new_init)

    return pruned, stats

# Compare unstructured vs structured at same target
target = 0.5
unstr_model, unstr_stats = prune_model(dense_model, target)
str_model, str_stats = structured_prune(dense_model, target)

# Accuracy comparison
x_test = np.random.randn(16, 128).astype(np.float32)
y_dense  = sess_dense.run(None, {'X': x_test})[0]
sess_unstr = ort.InferenceSession(unstr_model.SerializeToString(),
                                  providers=['CPUExecutionProvider'])
sess_str = ort.InferenceSession(str_model.SerializeToString(),
                                providers=['CPUExecutionProvider'])
y_unstr = sess_unstr.run(None, {'X': x_test})[0]
y_str   = sess_str.run(None, {'X': x_test})[0]

print(f"Comparison at {target:.0%} target sparsity:")
print(f"{'Method':<15} {'Mean|err|':>10} {'Max|err|':>10}")
print('-' * 38)
print(f"{'Unstructured':<15} {np.mean(np.abs(y_dense-y_unstr)):>10.6f} {np.max(np.abs(y_dense-y_unstr)):>10.6f}")
print(f"{'Structured':<15} {np.mean(np.abs(y_dense-y_str)):>10.6f} {np.max(np.abs(y_dense-y_str)):>10.6f}")

print(f"\nStructured pruning details:")
for s in str_stats:
    print(f"  {s['name']}: {s['cols_pruned']}/{s['cols_total']} columns pruned "
          f"({s['element_sparsity']:.1%} element sparsity)")

print(f"\nKey insight: Structured pruning gives higher error at the same sparsity ")
print(f"level because it removes entire feature dimensions, but it's easier to accelerate.")

---
## Exercise 7: Sparsity vs Accuracy Tradeoff Curve

The relationship between sparsity and accuracy degradation is typically:
- Low sparsity (0-50%): minimal accuracy loss
- Medium sparsity (50-80%): gradual degradation
- High sparsity (>90%): rapid accuracy collapse

This creates a "Pareto frontier" for the sparsity-accuracy tradeoff.

In [ ]:
sparsity_sweep = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]
errors_unstr = []
errors_str = []

n_eval = 50

for sp in sparsity_sweep:
    if sp == 0.0:
        errors_unstr.append(0.0)
        errors_str.append(0.0)
        continue

    u_model, _ = prune_model(dense_model, sp)
    s_model, _ = structured_prune(dense_model, sp)
    sess_u = ort.InferenceSession(u_model.SerializeToString(), providers=['CPUExecutionProvider'])
    sess_s = ort.InferenceSession(s_model.SerializeToString(), providers=['CPUExecutionProvider'])

    errs_u, errs_s = [], []
    for _ in range(n_eval):
        x = np.random.randn(16, 128).astype(np.float32)
        y_ref = sess_dense.run(None, {'X': x})[0]
        y_u = sess_u.run(None, {'X': x})[0]
        y_s = sess_s.run(None, {'X': x})[0]
        errs_u.append(np.mean(np.abs(y_ref - y_u) / (np.abs(y_ref) + 1e-8)))
        errs_s.append(np.mean(np.abs(y_ref - y_s) / (np.abs(y_ref) + 1e-8)))
    errors_unstr.append(np.mean(errs_u))
    errors_str.append(np.mean(errs_s))

print(f"{'Sparsity':>10} {'Unstr. RelErr':>15} {'Struct. RelErr':>15}")
print('-' * 42)
for sp, eu, es in zip(sparsity_sweep, errors_unstr, errors_str):
    print(f"{sp:>9.0%} {eu:>14.4%} {es:>14.4%}")

if HAS_MPL:
    plt.figure(figsize=(8, 5))
    plt.plot(sparsity_sweep, errors_unstr, 'o-', color='steelblue',
             label='Unstructured', linewidth=2, markersize=6)
    plt.plot(sparsity_sweep, errors_str, 's-', color='coral',
             label='Structured', linewidth=2, markersize=6)
    plt.xlabel('Sparsity Ratio', fontsize=12)
    plt.ylabel('Mean Relative Error', fontsize=12)
    plt.title('Sparsity vs Accuracy Tradeoff', fontweight='bold', fontsize=13)
    plt.legend(fontsize=11)
    plt.grid(alpha=0.3)
    plt.yscale('log')
    plt.tight_layout()
    plt.show()

---
## Challenge: Iterative Pruning Pipeline

In practice, pruning is done **iteratively**: prune a small amount, fine-tune,
repeat. This gradual approach preserves more accuracy than one-shot pruning.

Build a pipeline that:
1. Starts with a dense model
2. Prunes by incremental amounts
3. Reports sparsity and accuracy at each step
4. Identifies the maximum sparsity before accuracy collapses

In [ ]:
def iterative_prune_pipeline(model, steps, input_shape, n_eval=50, collapse_threshold=0.5):
    """Iteratively prune a model and track accuracy at each step.
    
    Args:
        steps: list of cumulative sparsity targets (e.g., [0.1, 0.2, ..., 0.9])
        collapse_threshold: relative error threshold for "accuracy collapse"
    """
    sess_ref = ort.InferenceSession(model.SerializeToString(),
                                   providers=['CPUExecutionProvider'])
    input_name = sess_ref.get_inputs()[0].name
    report = []
    collapse_point = None

    for target in steps:
        pruned, stats = prune_model(model, target)
        sess_p = ort.InferenceSession(pruned.SerializeToString(),
                                     providers=['CPUExecutionProvider'])

        # Compute actual global sparsity
        _, _, global_sp = compute_sparsity_profile(pruned)

        # Evaluate accuracy
        errors = []
        for _ in range(n_eval):
            x = np.random.randn(*input_shape).astype(np.float32)
            y_ref = sess_ref.run(None, {input_name: x})[0]
            y_p   = sess_p.run(None, {input_name: x})[0]
            errors.append(np.mean(np.abs(y_ref - y_p) / (np.abs(y_ref) + 1e-8)))

        mean_err = np.mean(errors)
        report.append({
            'target': target,
            'actual_sparsity': global_sp,
            'mean_rel_error': mean_err,
            'max_rel_error': np.max(errors),
        })

        if collapse_point is None and mean_err > collapse_threshold:
            collapse_point = target

    return report, collapse_point

steps = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.98]
report, collapse = iterative_prune_pipeline(
    dense_model, steps, (16, 128), n_eval=30, collapse_threshold=0.5
)

print("Iterative Pruning Report")
print("=" * 55)
print(f"{'Target':>8} {'Actual':>8} {'MeanRelErr':>12} {'MaxRelErr':>12} {'Status':>10}")
print('-' * 55)
for r in report:
    status = 'COLLAPSE' if r['mean_rel_error'] > 0.5 else 'OK'
    print(f"{r['target']:>7.0%} {r['actual_sparsity']:>7.1%} "
          f"{r['mean_rel_error']:>12.4%} {r['max_rel_error']:>12.4%} {status:>10}")

if collapse:
    print(f"\nAccuracy collapse detected at {collapse:.0%} sparsity.")
    safe_limit = steps[max(0, steps.index(collapse) - 1)]
    print(f"Recommended maximum sparsity: {safe_limit:.0%}")
else:
    print(f"\nNo accuracy collapse detected up to {steps[-1]:.0%} sparsity.")

---
## Summary

| Concept | What You Practiced |
|:---|:---|
| Magnitude pruning | Threshold $\tau = \text{percentile}(|W|, 100p)$, mask and zero |
| Sparsity measurement | $\text{sparsity} = 1 - \text{nnz}/\text{total}$ per-layer and global |
| Dense vs sparse | Compared outputs at multiple sparsity levels |
| Structured pruning | Column-wise by $\ell_2$-norm importance |
| Accuracy tradeoff | Measured error vs sparsity curve (Pareto frontier) |
| Iterative pruning | Pipeline with collapse detection |

**Key takeaway:** Unstructured pruning preserves accuracy better, but structured pruning
is easier to accelerate with hardware. Both require careful accuracy monitoring.

**Next:** [Benchmarking](../04_Benchmarking/)